# 🎬 ReelForge Render — ráp video MIỄN PHÍ trên Colab

Notebook này nhận **job ráp video** mà app ReelForge (app.thereelforge.com) đã gửi lên **Google Drive của chính bạn**, ráp bằng ffmpeg có sẵn trên Colab, rồi trả `FINAL.mp4` về Drive — app tự hiện tiến độ và kết quả.

**Cách dùng (2 bước):**
1. Menu **Runtime → Run all** (hoặc Ctrl+F9).
2. Khi Google hỏi quyền, chọn **đúng tài khoản Drive** bạn đang dùng trong app → **Allow**. Xong quay lại app chờ.

> Notebook chỉ đọc/ghi file do app ReelForge tạo trong Drive **của bạn** — không gửi dữ liệu đi đâu khác, không chứa khoá bí mật nào (mã nguồn công khai).

In [ ]:
#@title ▶️ Chạy ráp video (Run all là đủ — không cần sửa gì) { display-mode: "form" }
import io, json, os, re, subprocess, sys, time

NEWS_EXPORT_RAW = 'https://raw.githubusercontent.com/victorleworker/flowsmart-assets/main/colab/news_export.py'
WORK = '/content/render'
MIME_EXT = {'video/mp4':'mp4','video/webm':'webm','video/quicktime':'mov','image/png':'png',
            'image/jpeg':'jpg','image/webp':'webp','audio/mpeg':'mp3','audio/wav':'wav',
            'audio/mp4':'m4a','audio/ogg':'ogg','application/x-subrip':'srt','text/plain':'srt'}

print('① Đăng nhập Google (chọn đúng tài khoản Drive của app)…')
from google.colab import auth as colab_auth
colab_auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload, MediaInMemoryUpload
creds, _ = google.auth.default()
drive = build('drive', 'v3', credentials=creds)

def read_json(file_id):
    buf = io.BytesIO(); dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=file_id))
    done = False
    while not done: _, done = dl.next_chunk()
    return json.loads(buf.getvalue().decode('utf-8'))

def download(file_id, dest):
    req = drive.files().get_media(fileId=file_id)
    with open(dest, 'wb') as f:
        dl = MediaIoBaseDownload(f, req)
        done = False
        while not done: _, done = dl.next_chunk()

def find_folders(name, parent=None):
    # 'me' in owners: kẻo vớ nhầm thư mục trùng tên do người khác share vào Drive
    q = f"name='{name}' and mimeType='application/vnd.google-apps.folder' and trashed=false and 'me' in owners"
    if parent: q += f" and '{parent}' in parents"
    r = drive.files().list(q=q, fields='files(id)', pageSize=100).execute().get('files', [])
    return [f['id'] for f in r]

# Drive có thể có NHIỀU thư mục 'Reelforge' (vd. đã "Xóa dữ liệu Drive" rồi khôi phục bản cũ
# từ Thùng rác) → duyệt TẤT CẢ các cây Reelforge/render-queue, chọn cây thật sự có job đang chờ.
queues = [q for rf in find_folders('Reelforge') for q in find_folders('render-queue', rf)]
assert queues, 'Không thấy Reelforge/render-queue trên Drive — bấm "Gửi job + Mở Colab" trong app TRƯỚC, và đăng nhập ĐÚNG tài khoản Drive.'

print('② Tìm job đang chờ (mới nhất) trong Reelforge/render-queue…')
# App tạo SẴN status-<id>.json ('waiting') + FINAL-<id>.mp4 placeholder rồi ghi fileId vào
# job doc — notebook chỉ UPDATE 2 file đó (scope drive.file của app mới thấy được kết quả).
cands = []
for queue in queues:
    cands += drive.files().list(q=f"name contains 'job-' and '{queue}' in parents and trashed=false",
                                orderBy='createdTime desc', fields='files(id,name,createdTime)', pageSize=10).execute().get('files', [])
cands.sort(key=lambda f: f.get('createdTime', ''), reverse=True)  # gộp mọi queue, mới nhất trước
job = None
for f in cands:
    if not re.match(r'^job-.+\.json$', f['name']): continue
    d = read_json(f['id'])
    if not (d.get('manifest') and d.get('statusFileId') and d.get('finalFileId')): continue
    try: st = read_json(d['statusFileId'])
    except Exception: continue
    if st.get('status') == 'waiting': job = d; break
assert job, f'Không có job nào đang chờ (đã dò {len(queues)} thư mục render-queue) — bấm "Gửi job + Mở Colab" trong app rồi Run all lại.'
manifest = job['manifest']; job_id = job['jobId']
print(f"   → job {job_id} ({len(manifest.get('segments', []))} đoạn)")

def status(pct, step, **extra):
    st = extra.pop('status', 'running')
    body = {'status': st, 'pct': int(pct), 'step': step, **extra}
    media = MediaInMemoryUpload(json.dumps(body, ensure_ascii=False).encode('utf-8'), mimetype='application/json')
    # Ghi tiến độ phải CHỊU LỖI: Drive 500/429 thoáng qua KHÔNG được làm hỏng cả buổi render.
    # Trạng thái chốt (done/failed) cố nhiều lần hơn — app cần nó để kết thúc chờ.
    tries = 6 if st in ('done', 'failed') else 3
    for i in range(tries):
        try:
            drive.files().update(fileId=job['statusFileId'], media_body=media).execute()
            break
        except Exception as err:
            if i + 1 >= tries:
                if st in ('done', 'failed'): raise
                print(f'   ⚠ Bỏ qua 1 lần ghi tiến độ lỗi ({str(err)[:120]}) — vẫn render tiếp.')
                break
            time.sleep(1.5 * (i + 1))
    print(f'   {int(pct):3d}% {step}')

try:
    status(2, 'Chuẩn bị môi trường (ffmpeg + font)')
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-noto-core'], capture_output=True)
    os.makedirs(f'{WORK}/in', exist_ok=True); os.makedirs(f'{WORK}/out', exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(NEWS_EXPORT_RAW, f'{WORK}/news_export.py')

    # drive:// refs → tải về + thay path local (ext theo mime thật)
    refs = []
    def reg(o, k, fb):
        v = o.get(k)
        if isinstance(v, str) and v.startswith('drive://'): refs.append((o, k, v[8:], fb))
    for s in manifest.get('segments', []): reg(s, 'media', 'mp4' if s.get('type') == 'clip' else 'png')
    reg(manifest, 'voiceover', 'mp3'); reg(manifest, 'srt', 'srt')
    if manifest.get('music'): reg(manifest['music'], 'path', 'mp3')
    if manifest.get('watermark'): reg(manifest['watermark'], 'path', 'png')
    for o in (manifest.get('overlays') or []): reg(o, 'path', 'png')
    seen = {}; uniq = len({r[2] for r in refs})
    for o, k, fid, fb in refs:
        if fid not in seen:
            meta = drive.files().get(fileId=fid, fields='mimeType').execute()
            ext = MIME_EXT.get(meta.get('mimeType', ''), fb)
            seen[fid] = f'{WORK}/in/{fid}.{ext}'
            download(fid, seen[fid])
            status(2 + len(seen) / max(1, uniq) * 13, f'Tải tư liệu {len(seen)}/{uniq}')
        o[k] = seen[fid]
    manifest['outDir'] = f'{WORK}/out'
    with open(f'{WORK}/manifest.json', 'w') as f: json.dump(manifest, f)

    # chạy NGUYÊN news_export.py — parse SEG i/total / CONCAT / MIX / @@SUMMARY@@
    proc = subprocess.Popen([sys.executable, f'{WORK}/news_export.py', f'{WORK}/manifest.json'],
                            cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    summary = None; last = 0
    for line in proc.stdout:
        line = line.strip()
        m = re.match(r'^SEG (\d+)/(\d+)', line)
        now = time.time()
        if m and now - last > 2: status(15 + int(m.group(1)) / int(m.group(2)) * 65, f'Dựng đoạn {m.group(1)}/{m.group(2)}'); last = now
        elif line == 'CONCAT': status(82, 'Nối video')
        elif line == 'MIX': status(88, 'Trộn âm thanh + phụ đề')
        elif line.startswith('@@SUMMARY@@'): summary = json.loads(line[len('@@SUMMARY@@'):].strip())
    err_tail = proc.stderr.read()[-2000:]
    assert proc.wait() == 0 and summary, f'news_export lỗi: {err_tail[-500:]}'

    status(95, 'Đưa FINAL lên Drive')
    final = summary.get('final') or f'{WORK}/out/FINAL.mp4'
    # ĐỔ BYTES vào placeholder FINAL-<id>.mp4 app đã tạo (update, KHÔNG create — xem ghi chú trên)
    drive.files().update(fileId=job['finalFileId'],
                         media_body=MediaFileUpload(final, mimetype='video/mp4', resumable=True)).execute()
    status(100, 'Xong', status='done', durationSec=summary.get('durationSec', 0))
    print('\n✅ XONG — FINAL đã về Drive (Reelforge/Render). Quay lại app: video nằm trong Thư viện.')
except Exception as e:
    try:  # đừng bỏ mồ côi news_export.py chạy tốn VM khi render đã coi như hỏng
        if proc.poll() is None: proc.kill()
    except Exception: pass
    msg = str(e)[:400]
    try: status(0, '', status='failed', error=msg)
    except Exception: pass
    raise
